# Module 19: Parallel Processing — Solutions

## Complete Solutions for All Exercises

## Part 1: Threading and GIL — Solutions

In [ ]:
import threading
import multiprocessing
import time

# Exercise 1.1: GIL Demonstration
def cpu_heavy(n):
    total = 0
    for i in range(n):
        total += i ** 2
    return total

# Sequential
start = time.time()
for _ in range(4):
    cpu_heavy(10000000)
seq_time = time.time() - start

# Threading
start = time.time()
threads = []
for _ in range(4):
    t = threading.Thread(target=cpu_heavy, args=(10000000,))
    t.start()
    threads.append(t)
for t in threads:
    t.join()
thread_time = time.time() - start

# Multiprocessing
start = time.time()
processes = []
for _ in range(4):
    p = multiprocessing.Process(target=cpu_heavy, args=(10000000,))
    p.start()
    processes.append(p)
for p in processes:
    p.join()
mp_time = time.time() - start

print('===== GIL Demonstration =====')
print(f'Sequential:      {seq_time:.2f}s')
print(f'Threading:       {thread_time:.2f}s (GIL: no speedup)')
print(f'Multiprocessing: {mp_time:.2f}s (true parallelism)')
print(f'\nExplanation: GIL allows only one thread to execute bytecode at a time.')
print('Threads are good for I/O but not CPU-bound pure Python.')

In [ ]:
# Exercise 1.2: Thread Safety
counter = 0
lock = threading.Lock()

def increment_without_lock(n):
    global counter
    for _ in range(n):
        counter += 1  # Race condition

def increment_with_lock(n):
    global counter
    for _ in range(n):
        with lock:
            counter += 1

# Without lock (race condition)
counter = 0
threads = [threading.Thread(target=increment_without_lock, args=(100000,)) for _ in range(10)]
for t in threads: t.start()
for t in threads: t.join()
print(f'Without lock: counter = {counter} (expected: 1,000,000)')

# With lock
counter = 0
threads = [threading.Thread(target=increment_with_lock, args=(100000,)) for _ in range(10)]
for t in threads: t.start()
for t in threads: t.join()
print(f'With lock:    counter = {counter} (expected: 1,000,000)')

## Part 2: concurrent.futures — Solutions

In [ ]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import time

# Exercise 2.1: Parallel HTTP Fetches (Simulated)
def fetch_endpoint(eid):
    time.sleep(0.3)
    return f'data_{eid}'

endpoints = list(range(20))

# Sequential
start = time.time()
results_seq = [fetch_endpoint(e) for e in endpoints]
seq_time = time.time() - start

# ThreadPool 5 workers
start = time.time()
with ThreadPoolExecutor(max_workers=5) as ex:
    results_t5 = list(ex.map(fetch_endpoint, endpoints))
t5_time = time.time() - start

# ThreadPool 10 workers
start = time.time()
with ThreadPoolExecutor(max_workers=10) as ex:
    results_t10 = list(ex.map(fetch_endpoint, endpoints))
t10_time = time.time() - start

print('===== ThreadPoolExecutor Comparison =====')
print(f'Sequential (1 worker):    {seq_time:.2f}s ({20/seq_time:.1f} req/s)')
print(f'ThreadPool (5 workers):   {t5_time:.2f}s ({20/t5_time:.1f} req/s)')
print(f'ThreadPool (10 workers):  {t10_time:.2f}s ({20/t10_time:.1f} req/s)')

In [ ]:
# Exercise 2.2: Parallel File Processing
import random
import string

def generate_file_content():
    words = []
    for _ in range(random.randint(50, 200)):
        word = ''.join(random.choices(string.ascii_lowercase, k=random.randint(3, 10)))
        words.append(word)
    return ' '.join(words)

def process_text(text):
    words = text.split()
    return {
        'word_count': len(words),
        'unique_words': len(set(words)),
        'avg_word_length': sum(len(w) for w in words) / len(words) if words else 0
    }

# Generate 100 files
files = [generate_file_content() for _ in range(100)]

# Sequential
start = time.time()
results_seq = [process_text(f) for f in files]
seq_time = time.time() - start

# Parallel
start = time.time()
with ProcessPoolExecutor(max_workers=4) as ex:
    results_par = list(ex.map(process_text, files))
par_time = time.time() - start

print('===== Parallel File Processing =====')
print(f'Sequential: {seq_time:.3f}s')
print(f'Parallel (4 workers): {par_time:.3f}s')
print(f'Speedup: {seq_time / par_time:.1f}x')
print(f'Total words: {sum(r["word_count"] for r in results_par)}')

## Part 3: Multiprocessing — Solutions

In [ ]:
from multiprocessing import Pool, Manager
import math

# Exercise 3.1: Pool Variations
def factorial(n):
    return math.factorial(n)

def pair_sum(pair):
    return pair[0] + pair[1]

with Pool(processes=4) as pool:
    # map
    facts = pool.map(factorial, range(1, 11))
    print('map (factorials 1-10):', facts[:5], '...')
    
    # starmap
    pairs = [(1, 2), (3, 4), (5, 6), (7, 8)]
    sums = pool.starmap(pair_sum, pairs)
    print('starmap (pair sums):', sums)
    
    # imap (lazy)
    results = []
    for r in pool.imap(factorial, range(1, 6)):
        results.append(r)
        print(f'  imap yielded: {r}')
    
    # apply_async with callback
    result_holder = []
    def collect(r):
        result_holder.append(r)
    for i in range(5, 11):
        pool.apply_async(factorial, (i,), callback=collect)
    pool.close()
    pool.join()
    print('apply_async results:', result_holder)

In [ ]:
# Exercise 3.2: Shared State with Manager
def add_entries(shared_dict, proc_id, n=1000):
    for i in range(n):
        shared_dict[f'proc_{proc_id}_key_{i}'] = proc_id * 1000 + i

with Manager() as manager:
    shared = manager.dict()
    with Pool(processes=4) as pool:
        pool.starmap(add_entries, [(shared, i, 1000) for i in range(4)])
    
    print('===== Shared State with Manager =====')
    print(f'Total entries in shared dict: {len(shared)}')
    print(f'Expected: 4000')
    print(f'All present: {len(shared) == 4000}')

## Part 4: AsyncIO — Solutions

In [ ]:
import asyncio
import random

# Exercise 4.1: Async Web Scraper
async def scrape_page(page_id):
    delay = random.uniform(0.2, 0.5)
    await asyncio.sleep(delay)
    return f'Page {page_id} content (took {delay:.2f}s)'

async def run_scraper():
    tasks = [scrape_page(i) for i in range(15)]
    results = await asyncio.gather(*tasks)
    return results

start = time.time()
results = asyncio.run(run_scraper())
total = time.time() - start

print('===== Async Web Scraper =====')
print(f'Scraped 15 pages in {total:.2f}s')
print(f'Sequential would take: ~{15 * 0.35:.1f}s (average)')
print(f'Speedup: ~{15 * 0.35 / total:.1f}x')

In [ ]:
# Exercise 4.2: Async Producer-Consumer
async def producer(queue, producer_id, n_items):
    for i in range(n_items):
        item = f'item_{producer_id}_{i}'
        await queue.put(item)
        print(f'  Producer {producer_id}: produced {item}')
        await asyncio.sleep(0.1)

async def consumer(queue, consumer_id):
    while True:
        item = await queue.get()
        print(f'    Consumer {consumer_id}: processing {item}')
        await asyncio.sleep(0.2)
        queue.task_done()

async def run_pipeline():
    queue = asyncio.Queue()
    producers = [asyncio.create_task(producer(queue, i, 5)) for i in range(2)]
    consumers = [asyncio.create_task(consumer(queue, i)) for i in range(3)]
    
    await asyncio.gather(*producers)
    await queue.join()
    for c in consumers:
        c.cancel()
    print('Pipeline complete')

print('===== Async Producer-Consumer =====')
asyncio.run(run_pipeline())

## Part 5: ML Parallel Processing — Solutions

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from itertools import product
import time

# Exercise 5.1: Parallel Hyperparameter Search
X, y = make_classification(n_samples=2000, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

def train_and_score(params):
    rf = RandomForestClassifier(**params, random_state=42)
    rf.fit(X_train, y_train)
    score = rf.score(X_test, y_test)
    return params, score

all_params = [dict(zip(param_grid.keys(), values)) 
              for values in product(*param_grid.values())]
print(f'Total parameter combos: {len(all_params)}')

# Sequential
start = time.time()
seq_results = [train_and_score(p) for p in all_params]
seq_time = time.time() - start

# Parallel
start = time.time()
with ProcessPoolExecutor(max_workers=4) as ex:
    par_results = list(ex.map(train_and_score, all_params))
par_time = time.time() - start

print(f'\n===== Parallel Hyperparameter Search =====')
print(f'Sequential: {seq_time:.2f}s')
print(f'Parallel (4 workers): {par_time:.2f}s')
print(f'Speedup: {seq_time / par_time:.2f}x')

best = max(par_results, key=lambda x: x[1])
print(f'Best params: {best[0]}')
print(f'Best score: {best[1]:.4f}')

In [ ]:
import numpy as np

# Exercise 5.2: Parallel Batch Inference
X_large, _ = make_classification(n_samples=100000, n_features=20, random_state=42)

model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_large[:1000], np.random.randint(0, 2, 1000))

# Sequential
start = time.time()
preds_seq = model.predict(X_large)
seq_time = time.time() - start
seq_throughput = len(X_large) / seq_time

# Batch inference with ProcessPoolExecutor
def predict_batch(batch):
    from sklearn.ensemble import RandomForestClassifier
    import joblib
    # In practice, load model from file or use global
    return model.predict(batch)

n_batches = 8
batches = np.array_split(X_large, n_batches)

start = time.time()
with ProcessPoolExecutor(max_workers=8) as ex:
    preds_par = np.concatenate(list(ex.map(predict_batch, batches)))
par_time = time.time() - start
par_throughput = len(X_large) / par_time

print('===== Parallel Batch Inference =====')
print(f'Sequential:    {seq_time:.2f}s ({seq_throughput:,.0f} samples/s)')
print(f'Parallel (8):  {par_time:.2f}s ({par_throughput:,.0f} samples/s)')
print(f'Speedup: {seq_time / par_time:.2f}x')
print(f'Predictions match: {np.array_equal(preds_seq, preds_par)}')